In [4]:
import pandas as pd
import requests
import json

# requires json file with crafting recipes
# by william allen

In [5]:
bazaar_data = requests.get('https://api.hypixel.net/v2/skyblock/bazaar').json()
items_in_bz = set()

bz_lst = []

for item in bazaar_data['products']:
  items_in_bz.add(item)
  temp = list(bazaar_data['products'][item]['quick_status'].values())
  temp[5] = format(float(temp[5]))
  temp[0] = temp[0].strip()
  bz_lst.append(temp)

column_names = list(bazaar_data['products']['IRON_INGOT']['quick_status'].keys())

df_bz = pd.DataFrame(columns = column_names, data = bz_lst)

df_bz = df_bz.sort_values('productId')
df_bz

,productId,sellPrice,sellVolume,sellMovingWeek,sellOrders,buyPrice,buyVolume,buyMovingWeek,buyOrders
373,ABSOLUTE_ENDER_PEARL,12075.636635,358450,408542,35,13350.7,31902,134907,133
548,ACACIA_BIRDHOUSE,948547.953731,3068,386,232,1129960.1769911505,21538,4964,1196
726,AGARIMOO_TONGUE,7944.800000,23740,241630,28,10534.1,33724,193169,30
975,AGATHA_COUPON,9658.033223,204964,3082620,29,10868.272716044125,739198,893126,602
1522,ALLIGATOR_SKIN,305874.500000,3278,31309,17,357232.7488372093,3520,11521,24
...,...,...,...,...,...,...,...,...,...
1328,YELLOW_FLOWER,242.323521,8449608,1478887,276,299.9990430264289,233433,1150025,214
750,YOGGIE,64.900000,319724,390437,10,281.51725520137427,487420,189817,461
354,YOUNG_FRAGMENT,8107.700000,257628,690695,43,9322.727556440903,41264,297551,315
46,Z,88730.525864,130881,3860,101,169325.29666666666,1552,3198,21


In [6]:
craft_prices = []

item_data = json.load(open('InternalNameMappings.json', 'r'))
for item in item_data:
  working = True
  if item not in items_in_bz:
    continue
  if 'recipe' in item_data[item]:
    recipe = item_data[item]['recipe']
    craft_price = 0
    for slot in recipe:
      materials = str(recipe[slot])
      if ':' not in materials:
        continue
      colon_loc = materials.index(':')
      item_id = materials[:colon_loc].strip()
      count = int(materials[colon_loc + 1:].strip())
      try:
        price = df_bz.loc[df_bz['productId'] == item_id, 'buyPrice'].values[0]
      except IndexError:
        working = False

      craft_price += float(price) * count
      buy_price = df_bz.loc[df_bz['productId'] == item, 'buyPrice'].values[0]
      sell_price = df_bz.loc[df_bz['productId'] == item, 'sellPrice'].values[0]
      diff = sell_price - craft_price
      diff_p = diff / craft_price
    if working:
      craft_prices.append([item, buy_price, sell_price, craft_price, diff, diff_p])

cols = ['id', 'buy price', 'sell price', 'craft price', 'profit', 'profit (%)']
flips_df = pd.DataFrame(columns = cols, data = craft_prices)


flips_df





/tmp/ipython-input-359/1856710752.py:27: RuntimeWarning: invalid value encountered in scalar divide
  diff_p = diff / craft_price
/tmp/ipython-input-359/1856710752.py:27: RuntimeWarning: divide by zero encountered in scalar divide
  diff_p = diff / craft_price


,id,buy price,sell price,craft price,profit,profit (%)
0,ABSOLUTE_ENDER_PEARL,13350.7,12075.636635,2.527200e+04,-1.319636e+04,-0.522173
1,ACACIA_BIRDHOUSE,1129960.1769911505,948547.953731,9.968573e+05,-4.830934e+04,-0.048462
2,AMALGAMATED_CRIMSONITE_NEW,89998.69701492538,65001.312291,2.010423e+05,-1.360410e+05,-0.676678
3,AUTO_SMELTER,22613.70105820106,1000.115311,3.160926e+02,6.840227e+02,2.163995
4,BLAZE_WAX,271997.34837310197,18.300990,1.908580e+05,-1.908397e+05,-0.999904
...,...,...,...,...,...,...
228,WHALE_BAIT,3798.9153590322157,3103.302917,5.765175e+03,-2.661872e+03,-0.461716
229,WHEAT,8.215196314111203,3.000000,0.000000e+00,3.000000e+00,inf
230,WHIPPED_MAGMA_CREAM,203775.3422580645,195989.100000,2.809665e+05,-8.497736e+04,-0.302447
231,WINTER_WATER_ORB,6869999.4,900000.300000,5.485223e+06,-4.585223e+06,-0.835923


In [7]:
flips_df.sort_values('profit', ascending=False)

,id,buy price,sell price,craft price,profit,profit (%)
201,PRESUMED_GALLON_OF_RED_PAINT,1744222.1990196079,1.000000e+06,7.611418e+05,2.388583e+05,0.313816
60,ENCHANTED_HUGE_MUSHROOM_2,255990.26482813744,2.485050e+05,5.124832e+04,1.972567e+05,3.849037
59,ENCHANTED_HUGE_MUSHROOM_1,255198.5364460562,2.484467e+05,5.144757e+04,1.969991e+05,3.829124
194,NULL_BLADE,19299995.2,1.809290e+07,1.795117e+07,1.417310e+05,0.007895
38,ENCHANTED_COOKIE,77858.46550286608,7.400300e+04,6.665352e+04,7.349482e+03,0.110264
...,...,...,...,...,...,...
215,SOULFLOW_ENGINE,15740289.600000001,1.170147e+07,1.879779e+07,-7.096316e+06,-0.377508
218,STUFFED_CHILI_PEPPER,16998999.82857143,1.927292e+03,1.710089e+07,-1.709896e+07,-0.999887
175,INFERNO_VERTEX,669999.2646153846,6.450010e+05,1.948365e+07,-1.883865e+07,-0.966895
182,MAGMA_FISH_DIAMOND,10045244.393333334,9.000002e+06,4.284347e+07,-3.384347e+07,-0.789933
